# Lezione 03 — Distribuzioni empiriche

**Corso:** Statistica Applicata con Python · **Blocco 1:** Descrivere i dati
**Tempo stimato:** 90 minuti · **Capitolo del manuale:** `manual/03_distribuciones_empiricas_IT.md`

## Cosa vedrai in questa lezione

- Costruire e interpretare **istogrammi**, **KDE** (un "istogramma levigato"), **grafici a scatola** (boxplot) ed **ECDF**.
- Verificare con un **esperimento guidato** come il numero di intervalli (*bin*) e l'ampiezza di banda possano cambiare la storia.
- Scegliere il grafico in base alla domanda e **confrontare gruppi** senza ingannare (assi condivisi, senza nascondere la forma).
- Trattare in modo esplicito i **valori mancanti**.

> **Nota:** qui la KDE è presentata solo come istogramma levigato. Il concetto di **densità di probabilità** si formalizza nelle lezioni 4 e 5. Non usiamo test né intervalli di confidenza: descriviamo ciò che c'è nel campione.

## Prerequisiti

- Lezioni 1 e 2 (popolazione/campione; media, mediana, quartili) e capitolo 06 del corso di *Visualizzazione dei Dati con Python*.
- Librerie: `numpy`, `pandas`, `matplotlib`, `seaborn` (già installate in Google Colab; in locale: `pip install numpy pandas matplotlib seaborn`).

## I dati

Dataset `penguins` di `seaborn` (*Palmer Penguins*: Horst, Hill e Gorman): 344 pinguini di tre specie. Se non c'è una copia locale né connessione, il notebook genera un dataset **sintetico** con le stesse colonne (*esempio illustrativo*).

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seed**. Poi definiamo tre funzioni ausiliarie: caricare i dati (copia locale → download → sintetico), il generatore sintetico e `completos`, che **esclude i valori mancanti e dice quante righe toglie**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)          # seed fisso: otteniamo tutti gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
sns.set_theme(style="whitegrid")
print("Librerie caricate correttamente.")

In [ ]:
def generar_penguins_sintetico(semilla=2026):
    """Genera un dataset SINTETICO con le stesse colonne di `penguins`.

    ESEMPIO ILLUSTRATIVO: i valori imitano l'ordine di grandezza dei dati reali,
    ma NON sono pinguini reali. Serve solo come alternativa senza connessione."""
    r = np.random.default_rng(semilla)
    # per specie: n, (media, deviazione) di ogni misura e isole possibili
    especies = {
        "Adelie":    dict(n=152, pico=(38.8, 2.7), prof=(18.3, 1.2), aleta=(190, 6.5), masa=(3700, 460), islas=["Torgersen", "Biscoe", "Dream"]),
        "Chinstrap": dict(n=68,  pico=(48.8, 3.3), prof=(18.4, 1.1), aleta=(196, 7.1), masa=(3733, 384), islas=["Dream"]),
        "Gentoo":    dict(n=124, pico=(47.5, 3.1), prof=(15.0, 1.0), aleta=(217, 6.5), masa=(5076, 504), islas=["Biscoe"]),
    }
    bloques = []
    for esp, c in especies.items():
        n = c["n"]
        sexo = r.choice(["Male", "Female"], size=n)
        efecto = np.where(sexo == "Male", 0.5, -0.5)          # i maschi sono un po' più grandi delle femmine
        def medida(par):
            return par[0] + efecto * par[1] + r.normal(0, par[1] * 0.8, size=n)
        bloques.append(pd.DataFrame({
            "species": esp,
            "island": r.choice(c["islas"], size=n),
            "bill_length_mm": np.round(medida(c["pico"]), 1),
            "bill_depth_mm": np.round(medida(c["prof"]), 1),
            "flipper_length_mm": np.round(medida(c["aleta"])),
            "body_mass_g": np.round(medida(c["masa"]) / 25) * 25,     # la massa reale è arrotondata a 25 g
            "sex": sexo,
        }))
    d = pd.concat(bloques, ignore_index=True)
    idx = r.choice(len(d), size=11, replace=False)               # righe 0 e 1 senza alcuna misura; altre 9 righe senza sesso (come nei dati reali)
    d.loc[idx[:2], ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]] = np.nan
    d.loc[idx[2:], "sex"] = np.nan
    return d


def cargar_penguins():
    """Carica penguins: copia locale -> download da seaborn -> sintetico."""
    for ruta in ("datos/penguins.csv", "../datos/penguins.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "copia locale (" + ruta + ")"
    try:
        return sns.load_dataset("penguins"), "download da seaborn"
    except Exception:
        return generar_penguins_sintetico(), "SINTETICO (esempio illustrativo, non sono pinguini reali)"


def completos(df, columnas):
    """Restituisce le righe senza valori mancanti in `columnas` e INFORMA di quante ne esclude."""
    sub = df.dropna(subset=columnas)
    cols = ", ".join(columnas)
    print(f"[{cols}] righe usate: {len(sub)} su {len(df)}  (escluse per valori mancanti: {len(df) - len(sub)})")
    return sub

---
## 1. I dati e i loro valori mancanti

Prima di disegnare qualsiasi cosa: da dove vengono i dati e cosa manca? **Non li nascondiamo**: li contiamo.

In [ ]:
df, origen = cargar_penguins()
print("Origine dei dati:", origen)
print(f"Righe: {len(df)}   Colonne: {list(df.columns)}")

faltantes = df.isna().sum()
display(pd.DataFrame({"mancanti": faltantes, "% mancanti": (faltantes / len(df) * 100).round(1)}))
print("Righe con almeno un valore mancante:", df.isna().any(axis=1).sum())
display(df[df.isna().any(axis=1)].head(6))

**Decisione di trattamento:** non facciamo un `dropna()` globale. Per ogni grafico escludiamo solo le righe che **non hanno la variabile che disegniamo**, e lo diciamo (funzione `completos`). Così, una riga senza sesso conta comunque in un istogramma della massa corporea.

---
## 2. Istogramma: contare per intervalli

Prima, a mano e con i dodici dati del capitolo. Dividiamo l'intervallo in classi e contiamo quanti dati cadono in ciascuna.

In [ ]:
# I 12 dati di esempio del capitolo (esempio illustrativo)
x = np.array([2, 3, 7, 8, 8, 9, 12, 13, 13, 14, 21, 22])

bordes = [0, 5, 10, 15, 20, 25]      # estremi degli intervalli: [0,5) [5,10) [10,15) [15,20) [20,25]
conteos, _ = np.histogram(x, bins=bordes)
conteos10, _ = np.histogram(x, bins=[0, 10, 20, 30])

print("Conteggi con ampiezza 5 :", conteos)
print("Conteggi con ampiezza 10:", conteos10)
print("Somma dei conteggi:", conteos.sum(), "=", len(x))

**Osserva:** con ampiezza 5 si vede un vuoto tra 15 e 20 (conteggio 0). Con ampiezza 10 il vuoto scompare: `[10, 20)` contiene 12, 13, 13, 14 e nessun altro. Gli stessi dati, due storie.

Ora con la massa corporea reale dei pinguini: 5 intervalli calcolati da `numpy` e l'istogramma con 20.

In [ ]:
masa = completos(df, ["body_mass_g"])["body_mass_g"]

# Gli stessi passaggi con la massa corporea reale e 5 intervalli
conteos5, bordes5 = np.histogram(masa, bins=5)
for i in range(5):
    print(f"Intervallo [{bordes5[i]:.0f}, {bordes5[i+1]:.0f}):  conteggio = {conteos5[i]}")

fig, ax = plt.subplots()
sns.histplot(masa, bins=20, color="steelblue", ax=ax)
ax.set_title("Massa corporea dei pinguini (n = " + str(len(masa)) + ")")
ax.set_xlabel("Massa corporea (g)"); ax.set_ylabel("Numero di pinguini")
plt.show()

---
## 3. Esperimento guidato: quanti *bin*? quale ampiezza di banda?

Prendiamo la **lunghezza della pinna**, una variabile che mescola specie di dimensioni diverse. Disegniamo **gli stessi dati** con 3, 8, 20 e 60 intervalli, sullo stesso asse orizzontale. Prima di eseguire, annota: quanti picchi ti aspetti di vedere?

In [ ]:
aleta = completos(df, ["flipper_length_mm"])["flipper_length_mm"]

fig, ejes = plt.subplots(1, 4, figsize=(16, 3.4), sharex=True)
for ax, k in zip(ejes, [3, 8, 20, 60]):
    sns.histplot(aleta, bins=k, color="steelblue", ax=ax)
    ancho = (aleta.max() - aleta.min()) / k
    ax.set_title(f"{k} intervalli ({'ampiezza'} ≈ {ancho:.1f} mm)")
    ax.set_xlabel("Lunghezza della pinna (mm)"); ax.set_ylabel("Numero di pinguini")
plt.tight_layout(); plt.show()

**Cosa osservi?**

- Con quanti intervalli compare la separazione tra i gruppi? Con 3 si vede? Con 60 si vede meglio, o compare rumore?
- Quale caratteristica rimane stabile nei quattro grafici? È quella a cui puoi credere.
- Cosa sarebbe successo se avessi guardato solo il primo?

*(La tua risposta qui.)*

---
## 4. KDE: l'istogramma levigato

La KDE mette una piccola campana su ogni dato e le somma: una curva continua. Qui è solo un **istogramma levigato**; il suo significato come densità di probabilità arriva nelle lezioni 4 e 5. Il parametro chiave è l'**ampiezza di banda** (`bw_adjust` in `seaborn`). Proviamo tre valori sullo stesso istogramma.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True, sharey=True)
for ax, bw in zip(ejes, [0.2, 1, 3]):
    sns.histplot(aleta, bins=30, stat="density", color="lightsteelblue", ax=ax)
    sns.kdeplot(aleta, bw_adjust=bw, color="crimson", linewidth=2, ax=ax)
    ax.set_title(f"ampiezza di banda × {bw}")
    ax.set_xlabel("Lunghezza della pinna (mm)")
ejes[0].set_ylabel("Densità")
plt.tight_layout(); plt.show()

**Cosa osservi?**

- Con ampiezza di banda 0,2, quanti picchi disegna la curva? Sono tutti "reali"?
- Con 3, cosa è successo alla separazione tra i gruppi?
- In cosa scegliere l'ampiezza di banda assomiglia a scegliere il numero di *bin*?

*(La tua risposta qui.)*

### Attenzione ai bordi: la KDE esce dall'intervallo dei dati

`seaborn` prolunga la curva di alcune ampiezze di banda oltre il minimo e il massimo (`cut=3` per impostazione predefinita). Lo misuriamo con la pinna e lo vediamo con una variabile che **non può essere negativa**: dei tempi di attesa simulati *(esempio illustrativo)*.

In [ ]:
# La curva viene disegnata oltre il minimo e il massimo osservati
fig, ax = plt.subplots()
sns.kdeplot(aleta, ax=ax)
xs_kde = ax.lines[0].get_xdata()
plt.close(fig)
print("Intervallo dei dati della pinna        :", aleta.min(), "-", aleta.max())
print("Intervallo su cui viene disegnata la KDE:", round(xs_kde.min(), 1), "-", round(xs_kde.max(), 1))

# Tempi di attesa simulati (esempio illustrativo): non possono mai essere negativi
esperas = rng.exponential(scale=5, size=300)
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
sns.kdeplot(esperas, ax=ejes[0]);           ejes[0].set_title("Attesa: KDE predefinita")
sns.kdeplot(esperas, cut=0, ax=ejes[1]);    ejes[1].set_title("Attesa: KDE con cut=0")
for ax in ejes:
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Tempo di attesa (min)")
print("Minimo dei tempi di attesa:", round(esperas.min(), 3))
print("La KDE predefinita arriva fino a  :", round(ejes[0].lines[0].get_xdata().min(), 1))
plt.show()

**Cosa osservi?** La KDE predefinita disegna densità a sinistra dello 0, dove **non può esserci alcun dato**. Non è una previsione: è un effetto della levigatura. Con `cut=0` la curva viene tagliata nell'intervallo osservato. Regola: non leggere la KDE come "probabilità che esistano valori" fuori da ciò che è stato osservato.

---
## 5. Boxplot: compatto, ma senza forma

Il boxplot riassume mediana, quartili e valori anomali (lezione 2). Confrontiamo, per la stessa pinna, il boxplot con l'istogramma.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.4), sharex=True)
sns.boxplot(x=aleta, color="lightsteelblue", ax=ejes[0]);   ejes[0].set_title("Grafico a scatola (boxplot)")
sns.histplot(aleta, bins=25, color="steelblue", ax=ejes[1]);  ejes[1].set_title("Istogramma degli stessi dati")
for ax in ejes:
    ax.set_xlabel("Lunghezza della pinna (mm)")
ejes[0].set_ylabel("")
ejes[1].set_ylabel("Numero di pinguini")
plt.tight_layout(); plt.show()
print(aleta.describe().round(1))

**Cosa osservi?** La scatola è una sola, con mediana e quartili ragionevoli. Si intuisce in essa l'esistenza di due gruppi? L'istogramma sì. Il boxplot è ideale per **confrontare molti gruppi**; ma conviene sovrapporre i punti per non perdere la forma:

In [ ]:
datos_masa = completos(df, ["body_mass_g", "species"])

fig, ax = plt.subplots(figsize=(8, 3.6))
sns.boxplot(data=datos_masa, x="body_mass_g", y="species", color="lightsteelblue", fliersize=0, ax=ax)
sns.stripplot(data=datos_masa, x="body_mass_g", y="species", color="black", alpha=0.35, size=3, ax=ax)
ax.set_title("Massa corporea per specie"); ax.set_xlabel("Massa corporea (g)"); ax.set_ylabel("")
plt.show()

---
## 6. ECDF: la scala cumulata

La ECDF in *x* è la proporzione di dati ≤ *x*. La costruiamo a mano con i dodici dati e la confrontiamo con quella di `seaborn`.

In [ ]:
# ECDF a mano
x_ord = np.sort(x)
ecdf = np.arange(1, len(x_ord) + 1) / len(x_ord)
print("x   ECDF(x)")
for xv, yv in zip(x_ord, ecdf):
    print(f"{xv:>2}   {yv:.3f}")

masa_por_esp = completos(df, ["body_mass_g", "species"])
fig, ax = plt.subplots()
sns.ecdfplot(data=masa_por_esp, x="body_mass_g", hue="species", ax=ax)
ax.axvline(4000, color="gray", linestyle="--")
ax.set_title("ECDF della massa corporea per specie"); ax.set_xlabel("Massa corporea (g)"); ax.set_ylabel("Proporzione cumulata")
plt.show()

**Letture dirette:** sull'asse verticale si legge la proporzione al di sotto di un valore (linea tratteggiata a 4 000 g); in orizzontale, il percentile (dove la curva incrocia 0,5 c'è la mediana). Verifichiamo con i numeri:

In [ ]:
# Proporzione di pinguini di ogni specie con massa ≤ 4000 g
resumen = masa_por_esp.groupby("species")["body_mass_g"].agg(
    **{"proporzione ≤ 4000 g": lambda s: (s <= 4000).mean(), "mediana (g)": "median"}
).round(2)
display(resumen)

# Percentili letti sulla ECDF (quantili del campione)
print(masa_por_esp["body_mass_g"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

---
## 7. Scegliere il grafico in base alla domanda

| Domanda | Grafico |
|---|---|
| Che forma ha? Uno o più picchi? | Istogramma (provando vari *bin*) o KDE |
| Che proporzione sta sotto un valore? | ECDF |
| Come si confrontano centro e dispersione di molti gruppi? | Boxplot (con i punti) |
| Come differiscono le forme di pochi gruppi? | KDE o istogrammi con **lo stesso asse**, più ECDF |

## 8. Confrontare gruppi senza ingannare

Prima, l'errore classico: ogni pannello con la propria scala. Poi, la versione onesta: stessi assi e ogni gruppo normalizzato separatamente (`stat="density"`, `common_norm=False`).

In [ ]:
datos_aleta = completos(df, ["flipper_length_mm", "species"])

def panel_especies(compartir, titulo):
    fig, ejes = plt.subplots(1, 3, figsize=(13, 3.2), sharex=compartir, sharey=compartir)
    for ax, (esp, g) in zip(ejes, datos_aleta.groupby("species")):
        sns.histplot(g["flipper_length_mm"], bins=15, stat="density", ax=ax)
        ax.set_title(f"{esp} (n = {len(g)})")
        ax.set_xlabel("Lunghezza della pinna (mm)"); ax.set_ylabel("Densità")
    fig.suptitle(titulo)
    plt.tight_layout(); plt.show()

panel_especies(False, "Assi indipendenti (fuorviante)")
panel_especies(True, "Assi condivisi (onesto)")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 3.6), sharex=True)
sns.kdeplot(data=datos_aleta, x="flipper_length_mm", hue="species", common_norm=False, ax=ejes[0])
ejes[0].set_title("KDE per specie")
sns.ecdfplot(data=datos_aleta, x="flipper_length_mm", hue="species", ax=ejes[1])
ejes[1].set_title("ECDF per specie")
sns.boxplot(data=datos_aleta, x="flipper_length_mm", y="species", ax=ejes[2])
ejes[2].set_title("Boxplot per specie")
for ax in ejes:
    ax.set_xlabel("Lunghezza della pinna (mm)")
plt.tight_layout(); plt.show()

resumen_aleta = datos_aleta.groupby("species")["flipper_length_mm"].agg(
    **{"n": "count", "mediana": "median", "Q1": lambda s: s.quantile(0.25), "Q3": lambda s: s.quantile(0.75)}
)
display(resumen_aleta)

### La trappola di "media ± errore"

Due gruppi simulati *(esempio illustrativo)*: A con un solo picco e B miscela di due popolazioni. Le loro medie sono quasi uguali e le deviazioni simili, quindi le barre "media ± deviazione" **non distinguono** un gruppo dall'altro.

In [ ]:
# Due gruppi con la stessa media e deviazione simile, ma forme molto diverse (esempio illustrativo)
grupo_a = rng.normal(50, 11, size=400)
# B: miscela di due popolazioni, centrate su 40 e su 60
grupo_b = np.where(rng.random(400) < 0.5, rng.normal(40, 4, size=400), rng.normal(60, 4, size=400))

for nombre, g in [("A", grupo_a), ("B", grupo_b)]:
    print(f"{nombre}: media = {g.mean():.1f}   deviazione = {g.std():.1f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
ejes[0].bar(["Gruppo A", "Gruppo B"], [grupo_a.mean(), grupo_b.mean()], yerr=[grupo_a.std(), grupo_b.std()], capsize=8, color=["#4c72b0", "#dd8452"])
ejes[0].set_title("Barre: media ± deviazione"); ejes[0].set_ylabel("Valore")
sns.histplot(grupo_a, bins=30, alpha=0.6, label="Gruppo A", ax=ejes[1])
sns.histplot(grupo_b, bins=30, alpha=0.6, label="Gruppo B", ax=ejes[1])
ejes[1].set_title("Distribuzioni reali"); ejes[1].set_ylabel("Numero di pinguini"); ejes[1].legend()
plt.tight_layout(); plt.show()

**Cosa osservi?** Le barre di A e B sono quasi identiche; gli istogrammi non lo sono affatto. Un riassunto di due numeri non può sostituire la distribuzione. E attenzione: qui confrontiamo **guardando**; decidere se una differenza è reale o rumore richiede i metodi delle lezioni 6-9.

---
## Riepilogo della lezione

1. **Istogramma:** conta per intervalli; il numero di *bin* cambia la storia, quindi provane vari.
2. **KDE:** un istogramma levigato; l'ampiezza di banda ha il ruolo dei *bin*, e la curva esce dall'intervallo dei dati (la densità di probabilità si formalizza nelle lezioni 4 e 5).
3. **Boxplot:** compatto e adatto a confrontare molti gruppi, ma non mostra la forma.
4. **ECDF:** proporzione di dati ≤ *x*; senza parametri da regolare e ideale per confrontare gruppi.
5. **Confrontare gruppi:** stessi assi, ogni gruppo normalizzato separatamente, indicare l'*n* e i valori mancanti esclusi, e non sostituire la distribuzione con "media ± errore".

**Prossimo passo:** Lezione 4 — Probabilità per l'analista. E ora, mettiti alla prova: `notebooks_practica/clase03_practica_distribuciones_empiricas_IT.ipynb`.